# Heterogeneous Ensembles: Voting and Stacking

This notebook combines different model families for classification and regression. Synthetic data makes the examples reproducible.

## 1. Voting Classifier

In [ ]:
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, r2_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import VotingClassifier, VotingRegressor, StackingClassifier, StackingRegressor

X_cls, y_cls = make_classification(
    n_samples=500, n_features=20, n_informative=5,
    n_redundant=2, random_state=42
)
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.3, random_state=42, stratify=y_cls
)

lr = LogisticRegression(max_iter=1000, random_state=42)
svc = SVC(probability=True, random_state=42)
dtc = DecisionTreeClassifier(max_depth=3, random_state=42)

voting_clf = VotingClassifier(
    estimators=[("lr", lr), ("svc", svc), ("dtc", dtc)],
    voting="hard"
)
voting_clf.fit(X_train_cls, y_train_cls)
voting_pred = voting_clf.predict(X_test_cls)
print("Voting accuracy:", accuracy_score(y_test_cls, voting_pred))
print(classification_report(y_test_cls, voting_pred))

## 2. Voting Regressor

In [ ]:
X_reg, y_reg = make_regression(
    n_samples=500, n_features=20, n_informative=5,
    noise=10, random_state=42
)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

lin_reg = LinearRegression()
tree_reg = DecisionTreeRegressor(max_depth=3, random_state=42)
svr = SVR()

voting_reg = VotingRegressor(
    estimators=[("linear", lin_reg), ("tree", tree_reg), ("svr", svr)]
)
voting_reg.fit(X_train_reg, y_train_reg)
reg_pred = voting_reg.predict(X_test_reg)
print("Voting regressor MAE:", mean_absolute_error(y_test_reg, reg_pred))
print("Voting regressor R²:", r2_score(y_test_reg, reg_pred))

## 3. Stacking Classifier

In [ ]:
stacking_clf = StackingClassifier(
    estimators=[("lr", lr), ("svc", svc), ("dtc", dtc)],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5
)
stacking_clf.fit(X_train_cls, y_train_cls)
stack_pred = stacking_clf.predict(X_test_cls)
print("Stacking classifier accuracy:", accuracy_score(y_test_cls, stack_pred))
print(classification_report(y_test_cls, stack_pred))

## 4. Stacking Regressor

In [ ]:
stacking_reg = StackingRegressor(
    estimators=[
        ("linear", LinearRegression()),
        ("tree", DecisionTreeRegressor(max_depth=3, random_state=42)),
        ("svr", SVR())
    ],
    final_estimator=LinearRegression(),
    cv=5
)
stacking_reg.fit(X_train_reg, y_train_reg)
stack_reg_pred = stacking_reg.predict(X_test_reg)
print("Stacking regressor MAE:", mean_absolute_error(y_test_reg, stack_reg_pred))
print("Stacking regressor R²:", r2_score(y_test_reg, stack_reg_pred))

## Interpretation

Voting uses a fixed combination rule. Stacking learns a meta-model using cross-validated base-model predictions. The same train/test split is kept separate from meta-model training so final evaluation remains held out.